In [1]:
import sys
sys.path.append("..")

In [2]:
from src.rag import *
from src.preprocess import build_chunks
from src.config import EVALUATION_QUESTIONS
from src.model_client import get_client

/home/andrej-damjanovski/ml-study/ml-interns-mk-2026/interns/andrej/week_9/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


## Task 1

In [3]:
client = create_client()

In [4]:
collection = create_collection(client, "testings")

In [5]:
collection

Collection(name=testings)

In [6]:
texts, ids,metadatas = build_chunks(500,50)

In [8]:
add_documents(collection,texts,ids,metadatas)

In [9]:
results = query_collection(collection,"what LoRA rank did the lora paper use in its main GLUE experiments?")

In [10]:
results

{'ids': [['https-sebastianraschka-com-blog-2023-llm-finetuning-lora-html--9f533c97',
   'https-sebastianraschka-com-blog-2023-llm-finetuning-lora-html--5df6dcf8',
   'https-magazine-sebastianraschka-com-p-practical-tips-for-finetuning-llms--fd363254']],
 'embeddings': None,
 'documents': [['Implementing LoRA',
   "The implementation of LoRA is relatively straight-forward. We can think of it as a modified forward pass for the fully connected layers in an LLM. In pseudo-code, this looks like as follows:\ninput_dim = 768 # e.g., the hidden size of the pre-trained model\noutput_dim = 768 # e.g., the output size of the layer\nrank = 8 # The rank 'r' for the low-rank adaptation\n\nW = ... # from pretrained network with shape input_dim x output_dim",
   'weight += (lora_B @ lora_A) * scaling As we can see in the code formula above, the larger the influence of the LoRA weights.Previous experiments used r=8 and alpha=16, which resulted in a 2-fold scaling. Choosing alpha as two times r is a com

In [11]:
results["documents"][0][2]

'weight += (lora_B @ lora_A) * scaling As we can see in the code formula above, the larger the influence of the LoRA weights.Previous experiments used r=8 and alpha=16, which resulted in a 2-fold scaling. Choosing alpha as two times r is a common rule of thumb when using LoRA for LLMs, but I was curious if this still holds for larger r values. In other words, “alpha = 2×rank” really seems to be a sweet spot'

The clearest irrelevant result was the top raw-retrieval chunk. It explains a generic implementation but does not state the rank used in the original paper's main GLUE experiments. The other results mention `r=8` and `alpha=16`, or show a QLoRA GLUE table, but neither is evidence for the exact experiment asked about.

This was not mainly a vocabulary mismatch or a chunk boundary cutting the answer in half. It was an experiment and source ambiguity failure: the chunks share strong terms such as “LoRA,” “rank,” and “GLUE,” but they refer to a generic implementation, QLoRA, or later experiments rather than the original LoRA paper's main GLUE setup. The high distances in the raw results (about `0.92` to `0.99`) also indicate that these were weak semantic matches. The grounded model therefore did the right thing by abstaining instead of treating the nearby `r=8` value as the requested answer.

In [12]:
results = query_collection(collection,"What 4-bit data type does QLoRA introduce?")

In [13]:
results

{'ids': [['home-andrej-damjanovski-ml-study-ml-interns-mk-2026-interns-andrej-week-9-src-context-data-qlora-pdf--ccf61ffb',
   'home-andrej-damjanovski-ml-study-ml-interns-mk-2026-interns-andrej-week-9-src-context-data-qlora-pdf--78e8b7e7',
   'home-andrej-damjanovski-ml-study-ml-interns-mk-2026-interns-andrej-week-9-src-context-data-qlora-pdf--a0950d6e']],
 'embeddings': None,
 'documents': [['QLORA has one low-precision storage data type, in our case usually 4-bit, and one computation data\ntype that is usually BFloat16. In practice, this means whenever a QLORA weight tensor is used, we\ndequantize the tensor to BFloat16, and then perform a matrix multiplication in 16-bit.\nWe now discuss the components of QLORA followed by a formal definition of QLORA.\n4-bit NormalFloat Quantization\nThe NormalFloat (NF) data type builds on Quantile Quantization',
   'and (2) NF4 is superior to FP4 in terms of quantization precision.\nSummary\nOur results consistently show that 4-bit QLORA with NF4

In [14]:
def top_chunks(query_result):
    ids  = query_result["ids"][0]
    docs = query_result["documents"][0]
    return list(zip(ids, docs))

In [15]:
def show(question, ungrounded, grounded, context):
    line = "=" * 80
    print(line)
    print(f"Q: {question}")
    print(line)

    print("\n┌─ UNGROUNDED (no context) " + "─" * 40)
    print(indent(ungrounded))

    print("\n┌─ GROUNDED (with context) " + "─" * 40)
    print(indent(grounded))

    print("\n┌─ RETRIEVED SOURCES " + "─" * 46)
    for cid, text in context:
        print(f"  [{cid}]")
        print(indent(text[:200] + "...", pad=4))
    print("\n")

def indent(text, pad=2):
    prefix = " " * pad
    return "\n".join(prefix + line for line in text.strip().split("\n"))

In [16]:
model_client = get_client("openrouter", "meta-llama/llama-3.1-70b-instruct")

In [18]:
for question in EVALUATION_QUESTIONS:
    result = query_collection(collection,question)
    context = top_chunks(result)

    ungrounded = model_client(question)
    grounded_prompt = build_grounded_prompt(question, context)
    grounded = model_client(grounded_prompt)
    show(question,ungrounded,grounded,context)

Q: What LoRA rank r did the original LoRA paper use for its main GLUE experiments?

┌─ UNGROUNDED (no context) ────────────────────────────────────────
  A very specific question!
  
  Unfortunately, I'm a large language model, I don't have have access to the original LoRA paper or its experiments' details. However, I can suggest some possible ways to find the answer:
  
  1. Check the paper: If you have access to the original LoRA paper, you can check the experiments section or the appendix to see if the authors mention the LoRA rank used for their main GLUE experiments.
  2. Search online: You can try searching online for "LoRA paper GLUE experiments rank" or similar keywords to see if anyone has discussed or mentioned the LoRA rank used in the paper.
  3. Check the authors' website or repository: Sometimes, researchers make their code, data, or experiment details available on their website or GitHub repository. You can try checking the authors' website or repository to see if they p

The latest run shows that the main failure mode is the model **confidently making something up**, not only lacking knowledge. The ungrounded model claimed that LoRA reduced GPU memory by **11–14×**, while the retrieved source reports **33%**. It also claimed that the recommended scaling rule was `alpha ≈ 1/sqrt(r)`, while the retrieved source says `alpha = 2 * rank`. Those answers sound plausible but are unsupported and incorrect. By contrast, on the original-LoRA-rank and QLoRA-data-type questions, the ungrounded model admitted that it did not know; that is a lack of knowledge without hallucination.

These are not the same problem. **Not knowing** means the required information is missing from the model's usable knowledge; **hallucinating** means presenting an unsupported guess as a fact. RAG can help with both only when the corpus contains the answer and retrieval finds the correct chunk. RAG primarily reduces unsupported confident answers; it cannot fix missing knowledge that is not present in the retrieved context.

# Task 2 

In [3]:
from src.agents import create_agent
from src.observability import print_agent_trace, setup_langfuse_tracing, trace_agent_invocation

In [7]:
telemetry = setup_langfuse_tracing()
llama_agent = create_agent("meta-llama/llama-3.1-70b-instruct")

llama_task = """
Find Andrej's exam scores and calculate their arithmetic mean.
You must first retrieve the scores with the student-score lookup tool.
Then pass an expression made from the returned scores to the calculator tool.
Report the scores, the expression, and the mean.
"""

with trace_agent_invocation(llama_task, name="week-9-llama-baseline") as llama_span:
    llama_result = llama_agent(
        llama_task,
        limits={"turns": 5, "total_tokens": 8_000},
    )
    llama_span.set_attribute("output.value", str(llama_result))
    llama_trace_id = f"{llama_span.get_span_context().trace_id:032x}"

print("\nLlama final answer:")
print(llama_result)
llama_tools_used = print_agent_trace(llama_agent.messages)
print(f"Stop reason: {llama_result.stop_reason}")
print(f"Langfuse trace ID: {llama_trace_id}")
telemetry.tracer_provider.force_flush()



Tool #1: get_student_scores

Tool #2: calculator
First, Andrej's exam scores are: [92, 88, 96, 91].

Expression to calculate the mean of Andrej's exam scores: 
(sum(92, 88, 96, 91))/(4)

Result of calculating the mean: 91.75
Llama final answer:
First, Andrej's exam scores are: [92, 88, 96, 91].

Expression to calculate the mean of Andrej's exam scores: 
(sum(92, 88, 96, 91))/(4)

Result of calculating the mean: 91.75



=== Agent trajectory ===
1. user text: Find Andrej's exam scores and calculate their arithmetic mean.
First retrieve the scores, then calculate the mean from those returned scores.
Report the scores, expression, and result.
2. assistant text: 
3. assistant -> get_student_scores input={"student_name": "Andrej"}
4. assistant -> calculator input={"expression": "[get_student_scores(student_name=\"Andrej\")]", "mode": "evaluate", "precision": "null", "scientific": false, "force_numeric": false, "variables": "null", "wrt": "null", "point": "null", "order": "null"}
5. get_st

True

In [6]:
telemetry = setup_langfuse_tracing()
agent = create_agent("openai/gpt-4o-mini")

task = """
Find Andrej's exam scores and calculate their arithmetic mean.
You must first retrieve the scores with the student-score lookup tool.
Then pass an expression made from the returned scores to the calculator tool.
Report the scores, the expression, and the mean.
"""

with trace_agent_invocation(task) as trace_span:
    result = agent(task, limits={"turns": 5, "total_tokens": 8_000})
    trace_span.set_attribute("output.value", str(result))
    trace_span.set_attribute("output.mime_type", "text/plain")
    langfuse_trace_id = f"{trace_span.get_span_context().trace_id:032x}"

print("\nFinal answer:")
print(result)
tools_used = print_agent_trace(agent.messages)
print(f"Stop reason: {result.stop_reason}")
print(f"Langfuse trace ID: {langfuse_trace_id}")

# The OTLP exporter batches spans; flush before opening the Langfuse UI.
telemetry.tracer_provider.force_flush()


Tool #1: get_student_scores

Tool #2: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────────╮                                                                        │
│  │ Operation │ Evaluate Expression     │                                                                        │
│  │ Input     │ (92 + 88 + 96 + 91) / 4 │                                                                        │
│  │ Result    │ 91.75                   │                                                                        │
│  ╰───────────┴─────────────────────────╯                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Andrej's exam scores are: **92, 88, 96, 91**.

The arithmetic expression used to calculate the mean is: **(92 + 88 + 96 + 91) / 4**.

The arithmetic mean of the scores is: **91.75**.
Final answer:
Andrej's exam scores are: **92, 88, 96, 91**.

The arithmetic expression used to calculate the mean is: **(92 + 88 + 96 + 91) / 4**.

The arithmetic mean of the scores is: **91.75**.



=== Agent trajectory ===
1. user text: Find Andrej's exam scores and calculate their arithmetic mean.
You must first retrieve the scores with the student-score lookup tool.
Then pass an expression made from the returned scores to the calculator tool.
Report the scores, the expression, and the mean.
2. assistant -> get_student_scores input={"student_name": "Andrej"}
3. get_student_scores -> assistant output=[{"text": "[92, 88, 96, 91]"}]
4. assistant -> calculator input={"expression": "(92 + 88 + 96 + 91) / 4"}
5. calculator -> assistant output=[{"text": "Result: 91.75"}]
6. assistant text: Andrej's exam scores

True

The surprising moment was when the Llama agent called `calculator` in the same turn as `get_student_scores`. I expected it to call `get_student_scores`, wait for `[92, 88, 96, 91]`, and only then call `calculator`. Instead, it tried to pass `[get_student_scores(student_name="Andrej")]` as the calculator expression and supplied optional parameters as incorrectly typed `"null"` strings.

This suggests that the tools' descriptions explain each tool independently but do not make their dependency sufficiently clear for this model. The calculator's large schema also encouraged Llama to invent unnecessary arguments. A clearer description could state that calculator accepts only a completed mathematical expression and cannot invoke other tools inside that expression.

The surprising `calculator` span explains **what happened**: calculator was requested before the lookup result was available, it received malformed input and incorrectly typed optional parameters, and it returned validation errors.

It does not explain **why** the model chose that call because Llama emitted no reasoning text. This difference matters for debugging. **What happened** identifies the failing operation and invalid input, which locates the immediate technical problem. **Why it happened** identifies the underlying cause, such as an unclear description, a misunderstood dependency, or weak schema handling. The trace provides evidence for what happened; why it happened must be inferred by comparing the prompt, tool descriptions, and controlled runs with different models or descriptions.